In [0]:
import json, requests
from datetime import datetime

url="http://inceptezlabs.com/api.php"

ts = datetime.now().strftime("%Y%m%d%H%M%S")
sourcepath = f"/Volumes/prodcatalog/logistics/source/datalake_api/response_{ts}.json"

def read_api_json_file():
    try:
        response=requests.get(url)
        
        if response.status_code!=200:
            print(f"Error: {response.status_code}")
            return
        
        try:
            raw_json=response.json()
            data=json.dumps(raw_json)
            print(data)
        except json.JSONDecodeError:
            print("Error: The response is not valid JSON.")
            return
        
        try:
            dbutils.fs.put(sourcepath,data,overwrite=True)
            print(f"File written to {sourcepath}")
        except Exception as e:
            print(f"Error: {e}")
                            
    except Exception as e:
        print(f"The error message is: {e}")

read_api_json_file()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

user_schema = StructType([
    StructField("uid", StringType()),
    StructField("user", StructType([
        StructField("name", StringType()),
        StructField("email", StringType()),
        StructField("location", StringType()),
        StructField("registered", StringType()),])),])

raw_df=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format", "json")\
    .option("cloudFiles.schemaLocation", "/Volumes/prodcatalog/logistics/source/datalake_api/schema")\
    .option("cloudFiles.maxFilesPerTrigger",1)\
.load("/Volumes/prodcatalog/logistics/source/datalake_api/")

parsed_df=raw_df.withColumn("data",F.from_json(F.col("data"),user_schema))

struct_df=parsed_df.select(
    F.col("data.uid").alias("uid"),
    F.col("data.user.name").alias("name"),
    F.col("data.user.email").alias("email"),
    F.col("data.user.location").alias("location"),
    F.to_timestamp(F.col("data.user.registered")).alias("registered"),
    F.current_timestamp().alias("insertionTime")
)

struct_df.writeStream\
    .trigger(availableNow=True)\
    .format("delta")\
    .option("checkpointLocation", "/Volumes/prodcatalog/logistics/source/api_checkpoints/")\
.start("/Volumes/prodcatalog/logistics/bronze/streamwriter_api")

In [0]:
bronze_output=spark.read.format("delta").load("/Volumes/prodcatalog/logistics/bronze/streamwriter_api/")
display(bronze_output)